# Stage C2. Obtain candidate US biopharma job postings

The sample restrictions are applied in the following order:

1. Retain postings belonging to the Stage C1 company universe.
2. Retain US postings with a usable posting key, at least one usable raw or translated title,
   and an O*NET code whose first two characters are `17` or `19`. This is the same broad
   occupation-code rule used in Stage A and validated again in Stage B.
3. Retain postings inside each company's inclusive Stage C1 extraction window.
4. Normalize and screen raw and translated titles with the same title rules and rule version
   used in Stage B, then retain `is_occupation_eligible == True`.

To limit Fabric computation, Steps 1--4 operate only on the small set of columns needed for
filtering. The notebook then broadcasts only the eligible `job_id` values to filter the full
source without shuffling it. It writes all metadata and screening fields separately from the
keyed `description` text so local matching never needs to load the wide text column.


In [ ]:
"""
Task:
    Extract candidate US LinkedIn postings for the Stage B biopharma company universe.

Inputs:
(a) Fabric table `postings_linkedin`.
(b) Files/WenzhiW/A01_BaselineUSBiopharma/StageC1_CompanyWindows.parquet
    <== Constructed by StageC1_CompanyThresholdsForPostingDates.py.

Outputs:
(a) Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostings/
(b) Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostingText/
(c) Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostings.zip
(d) Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostingText.zip

Descriptions of outputs:
(1) Output (a) has one row per eligible posting key before hire-specific pairing. It keeps
    every source column except `description` and adds screening and window fields.
(2) Output (b) has the same posting keys and keeps only `id_job` and `description`.
(3) Outputs (c) and (d) package the corresponding Parquet directories for download.

Run:
    Import into Fabric, attach the source Lakehouse, place input (b) at the stated path,
    and run all cells in order.

Notes:
(1) This notebook has no dependency on any project-local Python module.
(2) First keep usable US postings in O*NET major groups 17 or 19; then apply the exact
    Stage B title rules to both raw and translated titles.
(3) The Stage C1 company universe already comes from US biopharma hires. Do not additionally
    filter posting rows on `rics_k400`, because that would silently change the company scope.
(4) Company-specific posting bounds are inclusive. Stage D later applies each hire's window.
(5) Preserve every source field across the two keyed outputs. Source fields are restored only
    after all restrictions are met.
(6) `job_category` is not treated as seniority; no posting seniority restriction is applied.
(7) Missing or duplicate eligible `job_id` values cause an error; records are not collapsed.
(8) `MONTH_ONLY_POLICY` governs month-only posting dates. Stage A assumes daily hire dates.
(9) Default output mode refuses replacement; set it to `overwrite` only intentionally.


Wang Wenzhi
Time: 2026-09-21
"""

import hashlib
import html
import json
from pathlib import Path
from zipfile import ZIP_STORED, ZipFile

import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, StringType, StructField, StructType


# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 1. Define the source, paths, schemas, and posting-date convention
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


POSTING_TABLE = "postings_linkedin"
INPUT_WINDOWS = (
    "Files/WenzhiW/A01_BaselineUSBiopharma/StageC1_CompanyWindows.parquet"
)
OUTPUT_METADATA = (
    "Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostings"
)
OUTPUT_TEXT = (
    "Files/WenzhiW/A01_BaselineUSBiopharma/StageC2_CandidateJobPostingText"
)
OUTPUT_WRITE_MODE = "errorifexists"
PARQUET_COMPRESSION = "zstd"
METADATA_PARTITIONS = 4
TEXT_PARTITIONS = 12
MONTH_ONLY_POLICY = "error"
LINK_RULE_VERSION = "company_calendar_1_12_inclusive_v1"
RULE_VERSION = "2026-09-21.1"
RAW_TITLE_COLUMN = "title_raw"
TRANSLATED_TITLE_COLUMN = "title_translated"
PUBLICATION_DATE_COLUMN = "post_date"
DESCRIPTION_COLUMN = "description"
MISSING_TEXT = ("", "empty", "null", "none", "nan", "na", "n/a")
DOCUMENTED_SOURCE_COLUMNS = [
    "job_id",
    "rcid",
    "company",
    "rics_k50",
    "rics_k200",
    "rics_k400",
    "title_raw",
    "title_translated",
    "job_category",
    "role_k50",
    "role_k150",
    "role_k300",
    "role_k500",
    "role_k1000",
    "role_k1250",
    "role_k1500",
    "location_raw",
    "region",
    "country",
    "state",
    "metro_area",
    "salary",
    "post_date",
    "remove_date",
    "ultimate_parent_rcid",
    "ultimate_parent_company_name",
    "onet_code",
    "onet_title",
    "remote_type",
    "jobtitle",
    "description",
    "salary_min",
    "salary_max",
    "salary_predicted",
]
INDEX_SOURCE_COLUMNS = [
    "job_id",
    "rcid",
    "country",
    "onet_code",
    RAW_TITLE_COLUMN,
    TRANSLATED_TITLE_COLUMN,
    PUBLICATION_DATE_COLUMN,
]
WINDOW_COLUMNS = [
    "id_rcid",
    "posting_lower",
    "posting_upper",
    "link_rule_version",
]
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


def clean_spark_text(column_name: str) -> Column:
    """
    Trim source text and recognize explicit missing tokens without changing the source.
    """
    value = F.trim(F.col(column_name).cast("string"))
    missing = value.isNull() | F.lower(value).isin(*MISSING_TEXT)
    return F.when(missing, F.lit(None).cast("string")).otherwise(value)


def add_source_date(source: DataFrame, column_name: str, output_name: str) -> DataFrame:
    """
    Add a parsed calendar date and a source-representation precision flag.

    Parameters
    ----------
    source : DataFrame
        Narrow posting-index frame; source fields are preserved.
    column_name : str
        Source field containing a date-like value.
    output_name : str
        Name assigned to the parsed date.

    Returns
    -------
    DataFrame
        Input rows with parsed date, precision, and month-policy fields.

    Notes
    -----
    (1) ISO daily strings retain their reported calendar day. Invalid dates become null.
    (2) Month-only strings require `MONTH_ONLY_POLICY`; no day is silently assumed.
    """
    value = clean_spark_text(column_name)
    is_month = value.rlike(r"^\d{4}-\d{2}$")
    is_day = value.rlike(r"^\d{4}-\d{2}-\d{2}(?:[ T].*)?$")
    if MONTH_ONLY_POLICY not in ("error", "first_day"):
        raise ValueError("MONTH_ONLY_POLICY must be error or first_day.")
    if MONTH_ONLY_POLICY == "error" and source.filter(is_month).limit(1).count():
        raise ValueError("Month-only dates found. Resolve the policy before extraction.")

    date_text = F.when(is_month, F.concat(value, F.lit("-01"))).when(
        is_day,
        F.substring(value, 1, 10),
    )
    helper_name = "_date_text_for_parsing"
    if helper_name in source.columns:
        raise ValueError(f"Reserved source column: {helper_name}.")
    parsed = source.withColumn(helper_name, date_text).withColumn(
        output_name,
        F.expr(f"try_cast(`{helper_name}` as date)"),
    )
    precision = (
        F.when(F.col(output_name).isNull(), F.lit("invalid_or_missing"))
        .when(is_month, F.lit("month"))
        .otherwise(F.lit("day_representation"))
    )
    return (
        parsed.withColumn(f"{output_name}_precision", precision)
        .withColumn("month_only_policy", F.lit(MONTH_ONLY_POLICY))
        .drop(helper_name)
    )


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 2. Define the self-contained Stage B title-screening rules
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


TITLE_ABBREVIATIONS = {
    "snr": "senior",
    "sr": "senior",
    "jr": "junior",
    "asst": "assistant",
    "assoc": "associate",
    "coord": "coordinator",
    "dir": "director",
    "engr": "engineer",
    "engg": "engineering",
    "exec": "executive",
    "mngr": "manager",
    "mgr": "manager",
    "supv": "supervisor",
    "dept": "department",
    "intl": "international",
    "mfg": "manufacturing",
    "mktg": "marketing",
    "mgmt": "management",
    "ops": "operations",
    "svp": "senior vice president",
    "evp": "executive vice president",
    "avp": "assistant vice president",
    "vp": "vice president",
    "ceo": "chief executive officer",
    "cfo": "chief financial officer",
    "coo": "chief operating officer",
    "cto": "chief technology officer",
    "cso": "chief scientific officer",
    "hr": "human resources",
    "qa": "quality assurance",
    "qc": "quality control",
    "cofounder": "co founder",
}


def normalize_job_titles(job_titles: pd.Series) -> pd.Series:
    """
    Normalize title form while preserving meaningful words and missing values.

    Parameters
    ----------
    job_titles : pd.Series
        Raw or translated titles, potentially containing missing values.

    Returns
    -------
    pd.Series
        Normalized titles with the original index and pandas string dtype.

    Notes
    -----
    (1) Work on distinct values to avoid repeating transformations for common titles.
    (2) Keep word order, scientific qualifiers, and seniority; do not use stemming.
    (3) Expand only explicit abbreviations. Ambiguous RA, AD, and PD are not expanded.
    """
    original = job_titles.astype("string")
    unique_titles = pd.Series(original.dropna().unique(), dtype="string")
    normalized = unique_titles.map(html.unescape).astype(
        pd.StringDtype(storage="python")
    )
    normalized = normalized.str.normalize("NFKC").str.casefold().str.strip()
    normalized = normalized.mask(normalized.isin(MISSING_TEXT))
    normalized = normalized.str.replace("_", " ", regex=False)
    normalized = normalized.str.replace(
        r"\br\s*(?:[&/+.\-]|and|\s)\s*d\b",
        "research and development",
        regex=True,
    )
    for dotted, expanded in (
        (r"\bq\.\s*a\.", "quality assurance"),
        (r"\bq\.\s*c\.", "quality control"),
        (r"\bc\.\s*r\.\s*a\.", "cra"),
        (r"\bv\.\s*p\.", "vice president"),
    ):
        normalized = normalized.str.replace(dotted, expanded, regex=True)
    normalized = normalized.str.replace("&", " and ", regex=False)
    normalized = normalized.str.replace(r"['\u2018\u2019\u02bc]", "", regex=True)
    normalized = normalized.str.replace(r"[^\w\s]", " ", regex=True)
    normalized = normalized.str.replace(r"\s+", " ", regex=True).str.strip()
    for abbreviation, replacement in TITLE_ABBREVIATIONS.items():
        normalized = normalized.str.replace(
            rf"\b{abbreviation}\b",
            replacement,
            regex=True,
        )
    normalized = normalized.mask(normalized.eq(""))
    lookup = pd.Series(normalized.array, index=unique_titles)
    return original.map(lookup).astype("string")


INTERNSHIP_PATTERN = (
    r"\b(?:interns?|internships?|externs?|externships?|co ops?|coops?|"
    r"co operative students?|cooperative students?|summer students?|working students?|"
    r"placement students?|work placements?|industrial placements?)\b"
)

EXCLUSION_PATTERNS = {
    "data_science": r"\bdata scien\w*\b",
    "clinical_operations": (
        r"\bclinical research (?:project |study )?(?:associates?|assistants?|"
        r"coordinators?|monitors?|managers?)\b|"
        r"\bclinical (?:trials?|study) (?:associates?|assistants?|coordinators?|"
        r"managers?|specialists?|administrators?|monitors?|leads?|leaders?|management|"
        r"operations)\b|\bclinical (?:operations|project|supply)\b|"
        r"\bclinical data (?:management|managers?|coordinators?|specialists?|"
        r"analysts?)\b|"
        r"\b(?:trial|study|site) (?:monitors?|coordinators?|managers?|management)\b|"
        r"\btrials? (?:leads?|leaders?|operations)\b|"
        r"\b(?:country approval|study start up|study startup|site activation|"
        r"site start up|patient recruitment|trial master file)\b|"
        r"\b(?:cra|cta|crc)\b"
    ),
    "quality_compliance": (
        r"\bquality (?:assurance|control|systems?|compliance|audit\w*|inspect\w*|"
        r"engineer\w*|specialists?|associates?|technicians?|analysts?|operations|"
        r"laboratory|scientists?|chemists?|leads?|leaders?)\b|"
        r"\b(?:product|supplier) quality\b|"
        r"\bcompliance (?:specialists?|officers?|analysts?|engineers?)\b|"
        r"\bgmp auditors?\b"
    ),
    "regulatory_medical_affairs": (
        r"\bregulatory (?:affairs|operations|submissions?|specialists?|associates?|"
        r"scientists?|coordinators?|managers?|writers?|writing)\b|"
        r"\bmedical affairs\b|\bmedical (?:science|scientific) liaisons?\b|"
        r"\bmsl\b|\bpharmacovigilance\b|\bdrug safety\b|"
        r"\b(?:medical|scientific) (?:writers?|writing|communications)\b"
    ),
    "operational_engineering": (
        r"\bcomputer systems? validation\b|\bcommissioning\b|"
        r"\bqualification engineers?\b|"
        r"\b(?:equipment validation|facilities|maintenance|field service|"
        r"technical support)\b|\breliability engineers?\b|"
        r"\b(?:manufacturing|production) "
        r"(?:associates?|technicians?|operators?|specialists?)\b|"
        r"\b(?:process|plant|chemical|bioprocess) operators?\b|"
        r"\b(?:supply chain|logistics|procurement|warehouse)\b"
    ),
    "commercial_administration": (
        r"\b(?:sales|marketing|business development|business analysts?|"
        r"human resources|talent acquisition|recruiters?|accountants?|finance|legal|"
        r"customer service|customer success)\b|"
        r"\bpatent (?:attorneys?|agents?|counsel)\b|"
        r"\b(?:administrative|executive) assistants?\b|\bsecretar\w*\b"
    ),
    "safety_healthcare": (
        r"\b(?:environmental health|occupational health|health and safety|ehs|hse)\b|"
        r"\bsafety (?:engineers?|specialists?|officers?)\b|"
        r"\bindustrial hygienists?\b|\b(?:nurses?|nursing|phlebotom\w*|"
        r"pharmacists?)\b|"
        r"\b(?:medical|clinical) (?:laboratory )?(?:technologists?|technicians?)\b"
    ),
    "it_business_analytics": (
        r"\bsoftware (?:development |test )?(?:engineers?|developers?|architects?)\b|"
        r"\b(?:data|cloud|network) (?:engineers?|developers?|architects?)\b|"
        r"\b(?:devops|cybersecurity|information technology|business intelligence)\b|"
        r"\bit (?:support|specialists?|analysts?|consultants?)\b|"
        r"\bsystems? administrators?\b"
    ),
    "nonemployment": (
        r"\b(?:retired|retirement|board members?|board observers?|advisory board|"
        r"chairman|chairwoman|chairperson|unemployed|open to work|"
        r"student ambassador|campus ambassador|volunteer)\b"
    ),
}

RESEARCH_ROLE_PATTERN = (
    r"\b(?:scientists?|chemists?|biochemists?|biologists?|microbiologists?|"
    r"immunologists?|virologists?|geneticists?|pharmacologists?|toxicologists?|"
    r"physicists?|bioengineers?|engineers?|investigators?|researchers?|"
    r"bioinformaticians?|postdocs?|postdoctoral)\b|"
    r"\bresearch (?:associates?|assistants?|fellows?|fellowships?|technicians?|"
    r"specialists?|scholars?|affiliates?)\b"
)

MANAGEMENT_PATTERN = (
    r"\b(?:managers?|directors?|head|vice president|president|chief|founders?|owners?|"
    r"supervisors?|consultants?|advisors?|advisers?)\b|"
    r"\b(?:team|group) leaders?\b"
)

RESEARCH_FUNCTION_PATTERNS = {
    "clinical_science": r"\bclinical (?:research )?scien\w*\b",
    "analytical_science": r"\b(?:analytical|bioanalytical)\b",
    "process_development": r"\b(?:process|bioprocess|cell line) development\b",
    "computational_biology": (
        r"\b(?:bioinformatics?|bioinformaticians?|cheminformatics?|"
        r"chemoinformatics?)\b|"
        r"\bcomputational (?:biology|biologists?|chemistry|chemists?|genomics)\b"
    ),
    "discovery_development": (
        r"\b(?:drug discovery|medicinal chemistry|formulation|protein engineering|"
        r"assay development|molecular biology|cell biology|pharmacology|toxicology|"
        r"immunology|research and development)\b"
    ),
}

QUALITY_DEVELOPMENT_PATTERN = (
    r"\b(?:analytical|bioanalytical|assay|method|methods|process|formulation) "
    r"development\b"
)

AMBIGUOUS_TECHNICAL_PATTERN = (
    r"\b(?:msat|cmc|validation|trainees?|apprentices?|students?|technicians?|"
    r"technologists?)\b|"
    r"\b(?:process|manufacturing|production|automation|project) engineers?\b|"
    r"^(?:(?:senior|principal|staff|associate|lead) )?engineer(?: [ivx]+| \d+)?$"
)

EXCLUSION_ORDER = [
    "missing_title",
    *EXCLUSION_PATTERNS,
    "management_without_research_role",
]


def classify_titles(normalized_titles: pd.Series) -> pd.DataFrame:
    """
    Classify normalized titles using functional evidence and explicit exceptions.

    Parameters
    ----------
    normalized_titles : pd.Series
        Normalized titles. Missing values are permitted.

    Returns
    -------
    pd.DataFrame
        Boolean exclusion and evidence flags indexed like the input.

    Notes
    -----
    (1) Analytical-development/QC work remains reviewable unless it is QA, audit, or
        quality-systems work.
    (2) Computing roles in computational biology remain eligible; data science does not.
    (3) Mixed researcher/manager titles remain reviewable; management without a research
        role is excluded.
    """
    flags = pd.DataFrame(index=normalized_titles.index)
    for reason, pattern in EXCLUSION_PATTERNS.items():
        flags[f"exclude_{reason}"] = normalized_titles.str.contains(pattern, na=False)
    flags["has_research_role"] = normalized_titles.str.contains(
        RESEARCH_ROLE_PATTERN,
        na=False,
    )
    flags["has_management_title"] = normalized_titles.str.contains(
        MANAGEMENT_PATTERN,
        na=False,
    )
    for function, pattern in RESEARCH_FUNCTION_PATTERNS.items():
        flags[f"is_{function}"] = normalized_titles.str.contains(pattern, na=False)
    function_columns = [
        f"is_{function}" for function in RESEARCH_FUNCTION_PATTERNS
    ]
    flags["has_research_function"] = flags[function_columns].any(axis=1)
    flags["is_mixed_management_title"] = (
        flags["has_management_title"] & flags["has_research_role"]
    )
    flags["exclude_management_without_research_role"] = (
        flags["has_management_title"] & ~flags["has_research_role"]
    )

    flag_quality_development = normalized_titles.str.contains(
        QUALITY_DEVELOPMENT_PATTERN,
        na=False,
    )
    flag_quality_assurance = normalized_titles.str.contains(
        r"\bquality (?:assurance|systems?|audit\w*|compliance)\b",
        na=False,
    )
    flags["is_mixed_quality_development"] = (
        flags["exclude_quality_compliance"]
        & flag_quality_development
        & ~flag_quality_assurance
    )
    flags["exclude_quality_compliance"] &= ~flags[
        "is_mixed_quality_development"
    ]

    flag_validation = normalized_titles.str.contains(r"\bvalidation\b", na=False)
    flag_scientific_validation = flag_validation & normalized_titles.str.contains(
        r"\b(?:analytical|bioanalytical|assay|method|methods)\b",
        na=False,
    )
    flag_validation_support = normalized_titles.str.contains(
        r"\b(?:engineers?|specialists?|analysts?|associates?|technicians?|"
        r"consultants?|coordinators?|contractors?|leads?|leaders?)\b",
        na=False,
    )
    flags["exclude_operational_engineering"] |= (
        flag_validation & flag_validation_support & ~flag_scientific_validation
    )
    flags["exclude_it_business_analytics"] &= ~flags[
        "is_computational_biology"
    ]
    flags["exclude_safety_healthcare"] &= ~normalized_titles.str.contains(
        r"\b(?:physician|nurse) scientists?\b",
        na=False,
    )
    flags["is_technical_title_ambiguous"] = normalized_titles.str.contains(
        AMBIGUOUS_TECHNICAL_PATTERN,
        na=False,
    ) & ~(
        normalized_titles.str.contains(
            QUALITY_DEVELOPMENT_PATTERN,
            na=False,
        )
        | flag_scientific_validation
    )
    return flags.astype(bool)


REVIEW_COLUMNS = [
    "is_mixed_management_title",
    "is_mixed_quality_development",
    "is_technical_title_ambiguous",
    "is_title_translation_conflict",
    "is_title_uninformative",
]


def title_rule_hash() -> str:
    """
    Return a stable hash of the Stage B title-rule configuration.
    """
    configuration = {
        "abbreviations": TITLE_ABBREVIATIONS,
        "internship": INTERNSHIP_PATTERN,
        "exclusions": EXCLUSION_PATTERNS,
        "research_role": RESEARCH_ROLE_PATTERN,
        "management": MANAGEMENT_PATTERN,
        "research_functions": RESEARCH_FUNCTION_PATTERNS,
        "quality_development": QUALITY_DEVELOPMENT_PATTERN,
        "ambiguous_technical": AMBIGUOUS_TECHNICAL_PATTERN,
        "exclusion_order": EXCLUSION_ORDER,
        "review_columns": REVIEW_COLUMNS,
    }
    encoded = json.dumps(
        configuration,
        sort_keys=True,
        separators=(",", ":"),
    ).encode()
    return hashlib.sha256(encoded).hexdigest()


def screen_titles(
    raw_titles: pd.Series,
    translated_titles: pd.Series,
) -> pd.DataFrame:
    """
    Return Stage B title decisions without modifying either source Series.

    Parameters
    ----------
    raw_titles, translated_titles : pd.Series
        Aligned source title fields.

    Returns
    -------
    pd.DataFrame
        Normalized titles, exclusions, review flags, and eligibility decisions.
    """
    raw = normalize_job_titles(raw_titles)
    translated = normalize_job_titles(translated_titles)
    flags = pd.DataFrame(index=raw.index)
    flags["title_raw_normalized"] = raw
    flags["title_translated_normalized"] = translated
    flags["job_title_normalized"] = raw.fillna(translated)
    flags["is_translated_title_missing"] = translated.isna()
    different = raw.notna() & translated.notna() & raw.ne(translated).fillna(False)
    flags["is_title_translation_different"] = different

    raw_flags = classify_titles(raw)
    translated_flags = classify_titles(translated)
    conflict = pd.Series(False, index=raw.index)
    for column in raw_flags:
        flags[column] = raw_flags[column] | translated_flags[column]
        if column.startswith("exclude_"):
            conflict |= raw_flags[column].ne(translated_flags[column])
    raw_internship = raw.str.contains(INTERNSHIP_PATTERN, na=False)
    translated_internship = translated.str.contains(INTERNSHIP_PATTERN, na=False)
    conflict |= raw_internship.ne(translated_internship)
    flags["is_title_translation_conflict"] = different & conflict
    flags["exclude_missing_title"] = flags["job_title_normalized"].isna()
    flags["is_title_uninformative"] = ~(
        flags["has_research_role"] | flags["has_research_function"]
    )
    flags["is_internship"] = raw_internship | translated_internship
    flags["needs_title_review"] = flags[REVIEW_COLUMNS].any(axis=1)
    flags["exclusion_reason"] = pd.Series(
        "retained",
        index=raw.index,
        dtype="string",
    )
    remaining = ~flags["is_internship"]
    flags.loc[~remaining, "exclusion_reason"] = "internship"
    for reason in EXCLUSION_ORDER:
        excluded = flags[f"exclude_{reason}"]
        flags.loc[remaining & excluded, "exclusion_reason"] = reason
        remaining &= ~excluded
    flags["is_occupation_eligible"] = remaining
    flags["is_strict_research_title"] = remaining & ~flags[
        "needs_title_review"
    ]
    flags["title_rule_version"] = RULE_VERSION
    flags["title_rule_hash"] = title_rule_hash()
    return flags


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 3. Validate the embedded title rules and define the Spark batch classifier
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


def run_title_regression_checks() -> None:
    """
    Exercise exclusion boundaries and raw-translation conflicts before scanning Fabric.
    """
    empty_titles = pd.Series([], dtype="string")
    if not screen_titles(empty_titles, empty_titles).empty:
        raise AssertionError("Empty-title regression check failed.")

    expected_reasons = {
        "data scientist": "data_science",
        "clinical research associate": "clinical_operations",
        "quality control analyst": "quality_compliance",
        "regulatory affairs scientist": "regulatory_medical_affairs",
        "manufacturing operator": "operational_engineering",
        "sales scientist": "commercial_administration",
        "safety engineer": "safety_healthcare",
        "software engineer": "it_business_analytics",
        "retired scientist": "nonemployment",
        "research intern": "internship",
        "chief scientific officer": "management_without_research_role",
        "QA analytical development scientist": "quality_compliance",
        "engineer II validation": "operational_engineering",
        "bioinformatics data scientist": "data_science",
    }
    retained = [
        "postdoctoral fellow",
        "clinical scientist",
        "QC analytical development scientist",
        "method validation scientist",
        "bioinformatics software engineer",
        "physician scientist",
        "nurse scientist",
        "scientist manager",
    ]
    titles = pd.Series([*expected_reasons, *retained, pd.NA], dtype="string")
    missing_translations = pd.Series(pd.NA, index=titles.index, dtype="string")
    results = screen_titles(titles, missing_translations)
    for index, expected_reason in enumerate(expected_reasons.values()):
        if results.loc[index, "exclusion_reason"] != expected_reason:
            title = titles.iloc[index]
            actual = results.loc[index, "exclusion_reason"]
            raise AssertionError(
                f"Title check failed for {title!r}: {actual!r}"
            )
    retained_start = len(expected_reasons)
    retained_end = retained_start + len(retained)
    if not results.iloc[retained_start:retained_end][
        "is_occupation_eligible"
    ].all():
        raise AssertionError("A retained-title boundary check failed.")
    if results.iloc[-1]["exclusion_reason"] != "missing_title":
        raise AssertionError("Missing-title regression check failed.")

    conflict = screen_titles(
        pd.Series(["scientist", "scientist"], dtype="string"),
        pd.Series(
            ["clinical research associate", "research intern"],
            dtype="string",
        ),
    )
    if not conflict["is_title_translation_conflict"].all():
        raise AssertionError("Translation-conflict regression check failed.")


run_title_regression_checks()
empty_title_flags = screen_titles(
    pd.Series(dtype="string"),
    pd.Series(dtype="string"),
)
title_schema = StructType(
    [
        StructField(
            column_name,
            BooleanType()
            if pd.api.types.is_bool_dtype(dtype)
            else StringType(),
            True,
        )
        for column_name, dtype in empty_title_flags.dtypes.items()
    ]
)


@F.pandas_udf(title_schema)
def posting_title_flags(
    raw_titles: pd.Series,
    translated_titles: pd.Series,
) -> pd.DataFrame:
    """
    Apply the embedded Stage B classifier to Arrow batches on Spark workers.
    """
    return screen_titles(raw_titles, translated_titles)


print(
    f"Validated embedded title rules {RULE_VERSION}; SHA-256: {title_rule_hash()}"
)


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 4. Validate the Stage C1 windows and the posting source schema
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


company_windows = spark.read.parquet(INPUT_WINDOWS)
missing_window_columns = sorted(
    set(WINDOW_COLUMNS) - set(company_windows.columns)
)
if missing_window_columns:
    raise ValueError(f"Missing company-window fields: {missing_window_columns}.")

invalid_windows = company_windows.filter(
    F.col("id_rcid").isNull()
    | F.col("posting_lower").isNull()
    | F.col("posting_upper").isNull()
    | (F.col("posting_lower") > F.col("posting_upper"))
    | ~F.col("link_rule_version").eqNullSafe(F.lit(LINK_RULE_VERSION))
)
if invalid_windows.limit(1).count():
    raise ValueError("Invalid company windows or incompatible linkage rule.")
duplicate_windows = (
    company_windows.groupBy("id_rcid")
    .count()
    .filter(F.col("count") != 1)
)
if duplicate_windows.limit(1).count():
    raise ValueError("Company-window keys are not unique.")

company_windows = company_windows.select(
    F.col("id_rcid").cast("string").alias("id_rcid"),
    F.col("posting_lower").cast("date").alias("posting_lower"),
    F.col("posting_upper").cast("date").alias("posting_upper"),
)

source_postings = spark.read.table(POSTING_TABLE)
source_columns = source_postings.columns
missing_source_columns = sorted(
    set(DOCUMENTED_SOURCE_COLUMNS) - set(source_columns)
)
if missing_source_columns or len(source_columns) != len(set(source_columns)):
    raise ValueError(
        "The posting table must have unique columns and all documented fields. "
        f"Missing fields: {missing_source_columns}."
    )
if source_postings.schema["rcid"].dataType.simpleString() not in ("int", "bigint"):
    raise TypeError("Recheck rcid: the documented source uses integral company IDs.")
if source_postings.schema["job_id"].dataType.simpleString() != "decimal(38,0)":
    raise TypeError("Recheck job_id: the documented source is decimal(38,0).")

generated_columns = [
    "id_rcid",
    "id_job",
    "publication_date",
    "publication_date_precision",
    "month_only_policy",
    "posting_lower",
    "posting_upper",
    *empty_title_flags.columns,
]
helper_columns = {"_title_flags", "_date_text_for_parsing"}
collisions = sorted(
    (set(generated_columns) | helper_columns) & set(source_columns)
)
if collisions:
    raise ValueError(f"Source fields collide with derived fields: {collisions}.")

print(
    f"Validated {len(source_columns)} posting source fields and the Stage C1 windows."
)


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 5. Restrict companies, country, occupation codes, and posting dates narrowly
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


# Read only seven small fields during restriction work. The wide source fields are not
# joined until Step 7, after occupation-code, date-window, and title restrictions.
posting_index = source_postings.select(*INDEX_SOURCE_COLUMNS).withColumn(
    "id_rcid",
    F.col("rcid").cast("string"),
)
company_postings = posting_index.join(
    F.broadcast(company_windows),
    on="id_rcid",
    how="inner",
)

is_usable_posting = F.col("job_id").isNotNull() & (
    clean_spark_text(RAW_TITLE_COLUMN).isNotNull()
    | clean_spark_text(TRANSLATED_TITLE_COLUMN).isNotNull()
)
is_us = clean_spark_text("country") == "United States"
is_focal_onet = F.substring(clean_spark_text("onet_code"), 1, 2).isin(
    "17",
    "19",
)
occupation_postings = company_postings.filter(
    is_usable_posting & is_us & is_focal_onet
)

dated_postings = add_source_date(
    occupation_postings,
    PUBLICATION_DATE_COLUMN,
    "publication_date",
)
window_postings = dated_postings.filter(
    F.col("publication_date").between(
        F.col("posting_lower"),
        F.col("posting_upper"),
    )
)
window_postings.explain("formatted")


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 6. Apply Stage B title rules and validate the eligible posting index
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


screened_index = (
    window_postings.withColumn(
        "_title_flags",
        posting_title_flags(
            F.col(RAW_TITLE_COLUMN),
            F.col(TRANSLATED_TITLE_COLUMN),
        ),
    )
    .select("*", "_title_flags.*")
    .drop("_title_flags")
)
candidate_index = (
    screened_index.filter(F.col("is_occupation_eligible"))
    .withColumn("id_job", F.col("job_id").cast("string"))
    .select("job_id", *generated_columns)
    .persist(StorageLevel.DISK_ONLY)
)

try:
    if candidate_index.filter(F.col("id_job").isNull()).limit(1).count():
        raise ValueError("Eligible postings contain missing job_id values.")
    duplicate_jobs = (
        candidate_index.groupBy("id_job")
        .count()
        .filter(F.col("count") > 1)
    )
    if duplicate_jobs.limit(1).count():
        raise ValueError(
            "Duplicate eligible job_id values found; resolve source records explicitly."
        )
    summary = (
        candidate_index.groupBy(
            "publication_date_precision",
            "needs_title_review",
        )
        .count()
        .collect()
    )
    posting_count = sum(row["count"] for row in summary)
    print(
        f"Eligible postings: {posting_count:,}; "
        f"date-precision/title-review summary: {summary}"
    )
except Exception:
    candidate_index.unpersist()
    raise


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 7. Restore source fields and write metadata and text separately
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


candidate_source = None
try:
    if OUTPUT_WRITE_MODE == "errorifexists":
        for output_path in [OUTPUT_METADATA, OUTPUT_TEXT]:
            destination = Path("/lakehouse/default") / output_path
            if destination.exists():
                raise FileExistsError(destination)
    candidate_keys = candidate_index.select("job_id")
    candidate_source = (
        source_postings.join(
            F.broadcast(candidate_keys),
            on="job_id",
            how="left_semi",
        )
        .persist(StorageLevel.DISK_ONLY)
    )
    candidate_source.explain("formatted")
    source_candidate_count = candidate_source.count()
    duplicate_source_jobs = (
        candidate_source.groupBy("job_id")
        .count()
        .filter(F.col("count") != 1)
    )
    if duplicate_source_jobs.limit(1).count():
        raise ValueError("Candidate job_id values are not unique in the source table.")
    if source_candidate_count != posting_count:
        raise ValueError(
            "Source-side candidate count differs from the eligible-key count: "
            f"{source_candidate_count:,} versus {posting_count:,}."
        )

    metadata_source_columns = [
        column for column in source_columns if column != DESCRIPTION_COLUMN
    ]
    candidate_metadata = (
        candidate_source.drop(DESCRIPTION_COLUMN)
        .join(candidate_index, on="job_id", how="inner")
        .select(*metadata_source_columns, *generated_columns)
    )
    candidate_text = candidate_source.select(
        F.col("job_id").cast("string").alias("id_job"),
        DESCRIPTION_COLUMN,
    )
    expected_metadata_columns = metadata_source_columns + generated_columns
    if candidate_metadata.columns != expected_metadata_columns:
        raise AssertionError("Stage C2 metadata fields are missing or out of order.")
    if candidate_text.columns != ["id_job", DESCRIPTION_COLUMN]:
        raise AssertionError("Stage C2 text fields are missing or out of order.")

    (
        candidate_metadata.repartition(METADATA_PARTITIONS)
        .write.mode(OUTPUT_WRITE_MODE)
        .option("compression", PARQUET_COMPRESSION)
        .parquet(OUTPUT_METADATA)
    )
    (
        candidate_text.repartition(TEXT_PARTITIONS)
        .write.mode(OUTPUT_WRITE_MODE)
        .option("compression", PARQUET_COMPRESSION)
        .parquet(OUTPUT_TEXT)
    )

    written_metadata = spark.read.parquet(OUTPUT_METADATA)
    written_text = spark.read.parquet(OUTPUT_TEXT)
    if written_metadata.columns != candidate_metadata.columns:
        raise AssertionError("Written Stage C2 metadata schema differs from the extract.")
    if written_text.columns != candidate_text.columns:
        raise AssertionError("Written Stage C2 text schema differs from the extract.")
    metadata_count = written_metadata.count()
    text_count = written_text.count()
    if metadata_count != posting_count or text_count != posting_count:
        raise AssertionError(
            "Written Stage C2 row counts differ from the eligible posting count."
        )
    print(
        f"Saved {metadata_count:,} metadata records with "
        f"{len(metadata_source_columns)} source fields: {OUTPUT_METADATA}"
    )
    print(f"Saved {text_count:,} keyed descriptions: {OUTPUT_TEXT}")
finally:
    if candidate_source is not None:
        candidate_source.unpersist()
    candidate_index.unpersist()


In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 8. Package both distributed Parquet outputs for download
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


def package_parquet_output(output_path: str) -> Path:
    """Package one Parquet directory without recompressing its encoded parts."""
    source_directory = Path("/lakehouse/default") / output_path
    archive_path = source_directory.with_suffix(".zip")
    staged_archive = archive_path.with_suffix(".zip.incomplete")
    if archive_path.exists() and OUTPUT_WRITE_MODE != "overwrite":
        raise FileExistsError(archive_path)
    source_files = sorted(
        path for path in source_directory.rglob("*.parquet") if path.is_file()
    )
    if not source_files:
        raise FileNotFoundError(f"No Parquet parts in {source_directory}.")

    with ZipFile(
        staged_archive,
        mode="w",
        compression=ZIP_STORED,
        allowZip64=True,
    ) as archive:
        for source_file in source_files:
            archive.write(
                source_file,
                arcname=source_file.relative_to(source_directory.parent).as_posix(),
            )
    with ZipFile(staged_archive) as archive:
        if archive.testzip() is not None:
            raise OSError(f"ZIP validation failed: {staged_archive}.")
    staged_archive.replace(archive_path)
    return archive_path


for output_path in [OUTPUT_METADATA, OUTPUT_TEXT]:
    packaged_path = package_parquet_output(output_path)
    print(
        f"Download and extract {packaged_path}; keep the directory and all Parquet parts."
    )
